In [31]:
import pandas as pd

df = pd.read_parquet("../data_processed/race_results_clean.parquet")
df = df.sort_values(['year', 'round', 'driverId']).reset_index(drop=True)
races = pd.read_csv("../Race data from 1950 to 2026 Race 9/races.csv", na_values=['\\N'])
print(df.shape)
df.head(3)

(27436, 33)


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,driverRef,forename,surname,nationality,constructorRef,name_constructor,nationality_constructor,status,quali_position,grid_fixed
0,20036,833,579,51,1.0,3.0,NaN,R,12,0.0,...,fangio,Juan,Fangio,Argentine,alfa,Alfa Romeo,Swiss,Oil leak,NaN,3.0
1,20042,833,589,105,19.0,11.0,NaN,R,18,0.0,...,chiron,Louis,Chiron,Monegasque,maserati,Maserati,Italian,Clutch,NaN,11.0
2,20030,833,619,151,12.0,13.0,6.0,6,6,0.0,...,gerard,Bob,Gerard,British,era,ERA,British,+3 Laps,NaN,13.0


feature 1: grid pposition

In [32]:
df['feat_grid'] = df['grid_fixed']
df[['grid', 'grid_fixed', 'feat_grid']].head(10)

,grid,grid_fixed,feat_grid
0,3.0,3.0,3.0
1,11.0,11.0,11.0
2,13.0,13.0,13.0
3,9.0,9.0,9.0
4,8.0,8.0,8.0
5,1.0,1.0,1.0
6,21.0,21.0,21.0
7,10.0,10.0,10.0
8,10.0,10.0,10.0
9,5.0,5.0,5.0


feature 2: recent form (rolling avg finishing position, last 3 races)

In [33]:
df = df.sort_values(['driverId', 'year', 'round'])

df['feat_driver_form_last3'] = (
    df.groupby('driverId')['positionOrder']
      .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

df[['driverId', 'year', 'round', 'positionOrder', 'feat_driver_form_last3']].head(15)

,driverId,year,round,positionOrder,feat_driver_form_last3
19241,1,2007,1,3,NaN
19263,1,2007,2,2,3.000000
19285,1,2007,3,2,2.500000
19307,1,2007,4,2,2.333333
19329,1,2007,5,2,2.000000
19351,1,2007,6,1,2.000000
19373,1,2007,7,1,1.666667
19395,1,2007,8,3,1.333333
19417,1,2007,9,3,1.666667
19439,1,2007,10,9,2.333333


feature 3: team pace

In [34]:
df = df.sort_values(['constructorId', 'year', 'round'])

df['feat_team_form_last3'] = (
    df.groupby('constructorId')['positionOrder']
      .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

df[['constructorId', 'year', 'round', 'positionOrder', 'feat_team_form_last3']].head(15)

,constructorId,year,round,positionOrder,feat_team_form_last3
3852,1,1968,8,13,NaN
3906,1,1968,11,8,13.000000
4424,1,1971,1,6,10.500000
4428,1,1971,1,23,9.000000
4438,1,1971,1,24,12.333333
4448,1,1971,2,5,17.666667
4453,1,1971,2,8,17.333333
4470,1,1971,3,4,12.333333
4475,1,1971,3,15,5.666667
4492,1,1971,4,12,9.000000


In [35]:
df['feat_team_form_last3'] = (
    df.groupby('constructorId')['positionOrder']
      .transform(lambda x: x.shift(1).rolling(window=6, min_periods=1).mean())
)

In [36]:
df = df.drop(columns=[c for c in df.columns if 'team_form' in c])

team_race_avg = (
    df.groupby(['constructorId', 'raceId', 'year', 'round'])['positionOrder']
      .mean()
      .reset_index()
      .rename(columns={'positionOrder': 'team_race_avg_position'})
)

team_race_avg = team_race_avg.sort_values(['constructorId', 'year', 'round'])
team_race_avg['feat_team_form_last3'] = (
    team_race_avg.groupby('constructorId')['team_race_avg_position']
      .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

df = df.merge(
    team_race_avg[['constructorId', 'raceId', 'feat_team_form_last3']],
    on=['constructorId', 'raceId'], how='left'
)

print([c for c in df.columns if 'team_form' in c])

['feat_team_form_last3']


In [37]:
df[['constructorId', 'year', 'round', 'positionOrder', 'feat_team_form_last3']].drop_duplicates().head(15)

,constructorId,year,round,positionOrder,feat_team_form_last3
0,1,1968,8,13,NaN
1,1,1968,11,8,13.000000
2,1,1971,1,6,10.500000
3,1,1971,1,23,10.500000
4,1,1971,1,24,10.500000
5,1,1971,2,5,12.888889
6,1,1971,2,8,12.888889
7,1,1971,3,4,10.722222
8,1,1971,3,15,10.722222
9,1,1971,4,12,11.222222


feature 4: circuit history

In [38]:
df = df.sort_values(['driverId', 'circuitId', 'year', 'round'])

df['feat_circuit_history'] = (
    df.groupby(['driverId', 'circuitId'])['positionOrder']
      .transform(lambda x: x.shift(1).expanding().mean())
)

df = df.sort_values(['year', 'round', 'driverId']).reset_index(drop=True)

df[['driverId', 'circuitId', 'year', 'round', 'positionOrder', 'feat_circuit_history']].head(15)

,driverId,circuitId,year,round,positionOrder,feat_circuit_history
0,579,9,1950,1,12,NaN
1,589,9,1950,1,18,NaN
2,619,9,1950,1,6,NaN
3,627,9,1950,1,5,NaN
4,640,9,1950,1,17,NaN
5,642,9,1950,1,1,NaN
6,660,9,1950,1,11,NaN
7,661,9,1950,1,20,NaN
8,666,9,1950,1,20,NaN
9,669,9,1950,1,14,NaN


In [39]:
sample = df[df['driverId'] == 1][['circuitId', 'year', 'round', 'positionOrder', 'feat_circuit_history']]
sample[sample['circuitId'] == sample['circuitId'].mode()[0]].head(15)

,circuitId,year,round,positionOrder,feat_circuit_history
19417,9,2007,9,3,NaN
19783,9,2008,9,1,3.000000
20123,9,2009,8,16,2.000000
20539,9,2010,10,2,6.666667
20971,9,2011,9,4,5.500000
21427,9,2012,9,8,5.200000
21869,9,2013,8,4,5.666667
22309,9,2014,9,1,5.428571
22698,9,2015,9,1,4.875000
23116,9,2016,10,1,4.444444


In [40]:
df[(df['driverId'] == 1) & (df['circuitId'] == 9)][['year', 'round', 'positionOrder', 'feat_circuit_history']]

,year,round,positionOrder,feat_circuit_history
19417,2007,9,3,NaN
19783,2008,9,1,3.000000
20123,2009,8,16,2.000000
20539,2010,10,2,6.666667
20971,2011,9,4,5.500000
21427,2012,9,8,5.200000
21869,2013,8,4,5.666667
22309,2014,9,1,5.428571
22698,2015,9,1,4.875000
23116,2016,10,1,4.444444


feature 5: reliability (DNF rate over the last 5 races)

In [41]:
df['dnf'] = df['position'].isna().astype(int)

df = df.sort_values(['driverId', 'year', 'round'])

df['feat_driver_dnf_rate_last5'] = (
    df.groupby('driverId')['dnf']
      .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

df = df.sort_values(['year', 'round', 'driverId']).reset_index(drop=True)

df[['driverId', 'year', 'round', 'position', 'dnf', 'feat_driver_dnf_rate_last5']].head(15)

,driverId,year,round,position,dnf,feat_driver_dnf_rate_last5
0,579,1950,1,NaN,1,NaN
1,589,1950,1,NaN,1,NaN
2,619,1950,1,6.0,0,NaN
3,627,1950,1,5.0,0,NaN
4,640,1950,1,NaN,1,NaN
5,642,1950,1,1.0,0,NaN
6,660,1950,1,11.0,0,NaN
7,661,1950,1,NaN,1,NaN
8,666,1950,1,NaN,1,NaN
9,669,1950,1,NaN,1,NaN


feature 6: points so far (championship standing going into this race)

In [42]:
driver_standings = pd.read_csv("../Race data from 1950 to 2026 Race 9/driver_standings.csv", na_values=['\\N'])

standings_sorted = driver_standings.sort_values(['driverId', 'raceId'])
standings_sorted['prev_points'] = standings_sorted.groupby('driverId')['points'].shift(1)
standings_sorted['prev_standing_position'] = standings_sorted.groupby('driverId')['position'].shift(1)

df = df.merge(
    standings_sorted[['raceId', 'driverId', 'prev_points', 'prev_standing_position']],
    on=['raceId', 'driverId'], how='left'
)

df[['driverId', 'year', 'round', 'prev_points', 'prev_standing_position']].head(15)

,driverId,year,round,prev_points,prev_standing_position
0,579,1950,1,31.0,1.0
1,589,1950,1,0.0,20.0
2,619,1950,1,0.0,40.0
3,627,1950,1,3.0,13.0
4,640,1950,1,2.0,16.0
5,642,1950,1,19.0,4.0
6,660,1950,1,0.0,26.0
7,661,1950,1,0.0,28.0
8,666,1950,1,0.0,74.0
9,669,1950,1,0.0,62.0


In [ ]:
driver_standings = pd.read_csv("../Race data from 1950 to 2026 Race 9/driver_standings.csv", na_values=['\\N'])

standings_with_time = driver_standings.merge(
    races[['raceId', 'year', 'round']], on='raceId', how='left'
)

standings_with_time = standings_with_time.sort_values(['driverId', 'year', 'round'])

standings_with_time['prev_points'] = standings_with_time.groupby('driverId')['points'].shift(1)
standings_with_time['prev_standing_position'] = standings_with_time.groupby('driverId')['position'].shift(1)

df = df.drop(columns=['prev_points', 'prev_standing_position'])

df = df.merge(
    standings_with_time[['raceId', 'driverId', 'prev_points', 'prev_standing_position']],
    on=['raceId', 'driverId'], how='left'
)

df[['driverId', 'year', 'round', 'prev_points', 'prev_standing_position']].head(15)

,driverId,year,round,prev_points,prev_standing_position
0,579,1950,1,NaN,NaN
1,589,1950,1,NaN,NaN
2,619,1950,1,NaN,NaN
3,627,1950,1,NaN,NaN
4,640,1950,1,NaN,NaN
5,642,1950,1,NaN,NaN
6,660,1950,1,NaN,NaN
7,661,1950,1,NaN,NaN
8,666,1950,1,NaN,NaN
9,669,1950,1,NaN,NaN


In [44]:
df[(df['driverId'] == 1) & (df['year'].isin([2007, 2008]))][
    ['year', 'round', 'points', 'prev_points', 'prev_standing_position']
].head(20)

,year,round,points,prev_points,prev_standing_position
19241,2007,1,6.0,NaN,NaN
19263,2007,2,8.0,6.0,3.0
19285,2007,3,8.0,14.0,3.0
19307,2007,4,8.0,22.0,3.0
19329,2007,5,8.0,30.0,1.0
19351,2007,6,10.0,38.0,2.0
19373,2007,7,10.0,48.0,1.0
19395,2007,8,6.0,58.0,1.0
19417,2007,9,6.0,64.0,1.0
19439,2007,10,0.0,70.0,1.0


In [45]:
feature_cols = [
    'feat_grid', 'feat_driver_form_last3', 'feat_team_form_last3',
    'feat_circuit_history', 'feat_driver_dnf_rate_last5',
    'prev_points', 'prev_standing_position'
]

df[feature_cols].isna().sum()

feat_grid                       20
feat_driver_form_last3         865
feat_team_form_last3           352
feat_circuit_history          8311
feat_driver_dnf_rate_last5     865
prev_points                   1331
prev_standing_position        1344
dtype: int64

In [46]:
print(df['year'].min(), df['year'].max())
print(df['year'].value_counts().sort_index())

1950 2026
year
1950    160
1951    179
1952    215
1953    246
1954    230
       ... 
2022    440
2023    440
2024    479
2025    479
2026    198
Name: count, Length: 77, dtype: int64


In [47]:
train = df[df['year'] <= 2018].copy()
val = df[(df['year'] >= 2019) & (df['year'] <= 2021)].copy()
test = df[(df['year'] >= 2022) & (df['year'] <= 2025)].copy()
future_2026 = df[df['year'] == 2026].copy()

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)
print("2026 (reserved for final prediction):", future_2026.shape)

Train: (24200, 41)
Validation: (1200, 41)
Test: (1838, 41)
2026 (reserved for final prediction): (198, 41)


In [48]:
feature_cols = [
    'feat_grid', 'feat_driver_form_last3', 'feat_team_form_last3',
    'feat_circuit_history', 'feat_driver_dnf_rate_last5',
    'prev_points', 'prev_standing_position'
]

# Compute fill values using ONLY the training set
fill_values = train[feature_cols].median()
print(fill_values)

feat_grid                     13.000000
feat_driver_form_last3        12.666667
feat_team_form_last3          12.666667
feat_circuit_history          12.000000
feat_driver_dnf_rate_last5     0.400000
prev_points                    3.000000
prev_standing_position        13.000000
dtype: float64


In [49]:
for split_df in [train, val, test, future_2026]:
    split_df[feature_cols] = split_df[feature_cols].fillna(fill_values)

# Confirm no NaNs remain in any split
for name, split_df in [('train', train), ('val', val), ('test', test), ('future_2026', future_2026)]:
    print(name, split_df[feature_cols].isna().sum().sum())

train 0
val 0
test 0
future_2026 0


In [52]:
train.to_parquet("../data_processed/train.parquet", index=False)
val.to_parquet("../data_processed/val.parquet", index=False)
test.to_parquet("../data_processed/test.parquet", index=False)
future_2026.to_parquet("../data_processed/future_2026.parquet", index=False)

print("All splits saved.")

All splits saved.
